# K-means Segmentation

**Project question:** Do standardized customer profiles contain useful exploratory segments?

By the end of this notebook, you should be able to:

- compare candidate K values with inertia and silhouette heuristics
- profile cluster sizes and raw-feature means before naming clusters
- state why cluster labels and validation scores do not prove real populations

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [ ]:
df = pd.read_csv(DATA / 'simulated_customer_profiles.csv')
features = ['visits_per_month', 'avg_order_value', 'discount_rate', 'email_opens', 'tenure_months', 'support_contacts', 'returns_per_year']
X_scaled = StandardScaler().fit_transform(df[features])

In [ ]:
rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=4031, n_init=20)
    labels = km.fit_predict(X_scaled)
    rows.append({'k': k, 'inertia': km.inertia_, 'silhouette': silhouette_score(X_scaled, labels)})
summary = pd.DataFrame(rows)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(summary['k'], summary['inertia'], marker='o')
axes[0].set_xlabel('K')
axes[0].set_title('Elbow plot')
axes[1].plot(summary['k'], summary['silhouette'], marker='o')
axes[1].set_xlabel('K')
axes[1].set_title('Silhouette by K')
plt.tight_layout()

Inertia always decreases as K increases. Silhouette is a useful separation heuristic, not proof of a uniquely correct number of clusters. Domain usefulness and stability under reasonable perturbations also matter.

In [ ]:
selected_k = int(summary.loc[summary['silhouette'].idxmax(), 'k'])
km = KMeans(n_clusters=selected_k, random_state=4031, n_init=20)
df['cluster'] = km.fit_predict(X_scaled)
cluster_sizes = df['cluster'].value_counts().sort_index().rename('n')
profile = df.groupby('cluster')[features].mean().round(2)
cluster_sizes, profile

In [ ]:
teaching_validation = adjusted_rand_score(df['true_segment'], df['cluster'])
print(f'Adjusted Rand index against the hidden synthetic segment: {teaching_validation:.3f}')

**Interpretation:** The synthetic `true_segment` was not used to fit or select the clustering; it is revealed only for teaching validation. Real segmentation projects usually lack such ground truth. Cluster numbers are arbitrary and may permute across fits.

**Transfer exercise:** Propose a substantive name for each cluster only after reading its profile, then write one sentence explaining why the name is provisional. Repeat the fit with another seed and check whether the profiles remain recognizable.